# Chapter 2: Supervised Learning

**Sumber:** Introduction to Machine Learning with Python – Andreas C. Müller & Sarah Guido

## Tujuan Praktikum
Pada chapter ini, kita akan mempelajari berbagai algoritma **Supervised Learning** yang mencakup:
- Classification (KNN, Linear Models, Naive Bayes, Decision Tree, Ensemble Methods, SVM, Neural Networks)
- Regression (Linear Regression, Ridge, Lasso, KNN Regressor)
- Evaluasi model dan pemahaman konsep generalization, overfitting, dan underfitting
- Uncertainty estimates dari classifier (decision_function dan predict_proba)


## 1. Persiapan Environment

In [ ]:
# Install library yang diperlukan
!pip install mglearn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import mglearn
from sklearn.model_selection import train_test_split

# Pengaturan tampilan matplotlib
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 2. Classification and Regression

Dua jenis utama supervised learning:
- **Classification**: memprediksi label kelas (diskrit)
- **Regression**: memprediksi nilai kontinu


## 3. Generalization, Overfitting, and Underfitting

Model yang baik harus mampu **generalize** ke data baru yang belum pernah dilihat sebelumnya.
- **Overfitting**: model terlalu kompleks, hafal data training tapi buruk di test
- **Underfitting**: model terlalu sederhana, buruk di training dan test


## 4. Sample Datasets

Buku menggunakan beberapa dataset sintesis dan nyata:
- `make_forge()` – dataset klasifikasi sederhana
- `make_wave()` – dataset regresi sederhana
- Wisconsin Breast Cancer dataset
- Boston Housing dataset (diganti dengan California Housing karena deprecation)


In [ ]:
# Dataset forge - untuk klasifikasi
X, y = mglearn.datasets.make_forge()
print("X.shape:", X.shape)
mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
plt.legend(["Class 0", "Class 1"], loc=4)
plt.xlabel("First feature")
plt.ylabel("Second feature")
plt.title("Figure 2-2: Scatter plot of the forge dataset")
plt.show()

In [ ]:
# Dataset wave - untuk regresi
X, y = mglearn.datasets.make_wave(n_samples=40)
plt.plot(X, y, 'o')
plt.ylim(-3, 3)
plt.xlabel("Feature")
plt.ylabel("Target")
plt.title("Figure 2-3: Plot of the wave dataset")
plt.show()

In [ ]:
# Breast Cancer dataset
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
print("cancer.keys():\n{}".format(cancer.keys()))

In [ ]:
print("Shape of cancer data: {}".format(cancer.data.shape))

In [ ]:
print("Sample counts per class:\n{}".format(
    {n: v for n, v in zip(cancer.target_names, np.bincount(cancer.target))}))

In [ ]:
print("Feature names:\n{}".format(cancer.feature_names))

In [ ]:
# Extended Boston Housing dataset (diganti California Housing karena load_boston deprecated)
# Buku menggunakan load_boston(), namun sudah dihapus di scikit-learn >= 1.2
# Kita gunakan dataset alternatif yang strukturnya serupa
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
print("Data shape: {}".format(housing.data.shape))
print("Feature names: {}".format(housing.feature_names))

In [ ]:
# Extended boston dari mglearn (menggunakan boston internal mglearn)
X, y = mglearn.datasets.load_extended_boston()
print("X.shape: {}".format(X.shape))

## 5. k-Nearest Neighbors (KNN)

KNN adalah salah satu algoritma ML paling sederhana. Model hanya menyimpan data training.
Untuk prediksi, algoritma mencari k tetangga terdekat dari data baru.


### 5.1 KNN Classification

In [ ]:
mglearn.plots.plot_knn_classification(n_neighbors=1)
plt.title("Figure 2-4: Predictions made by the one-nearest-neighbor model")
plt.show()

In [ ]:
mglearn.plots.plot_knn_classification(n_neighbors=3)
plt.title("Figure 2-5: Predictions made by the three-nearest-neighbors model")
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X, y = mglearn.datasets.make_forge()

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

print("Test set predictions: {}".format(clf.predict(X_test)))
print("Test set accuracy: {:.2f}".format(clf.score(X_test, y_test)))

#### Decision Boundary KNN dengan berbagai nilai n_neighbors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for n_neighbors, ax in zip([1, 3, 9], axes):
    clf = KNeighborsClassifier(n_neighbors=n_neighbors).fit(X, y)
    mglearn.plots.plot_2d_separator(clf, X, fill=True, eps=0.5, ax=ax, alpha=.4)
    mglearn.discrete_scatter(X[:, 0], X[:, 1], y, ax=ax)
    ax.set_title("{} neighbor(s)".format(n_neighbors))
    ax.set_xlabel("feature 0")
    ax.set_ylabel("feature 1")
axes[0].legend(loc=3)
plt.suptitle("Figure 2-6: Decision boundaries for different values of n_neighbors")
plt.tight_layout()
plt.show()

#### Model Complexity - Efek n_neighbors terhadap training dan test accuracy

In [ ]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=66)

training_accuracy = []
test_accuracy = []
# try n_neighbors from 1 to 10
neighbors_settings = range(1, 11)

for n_neighbors in neighbors_settings:
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    clf.fit(X_train, y_train)
    training_accuracy.append(clf.score(X_train, y_train))
    test_accuracy.append(clf.score(X_test, y_test))

plt.plot(neighbors_settings, training_accuracy, label="training accuracy")
plt.plot(neighbors_settings, test_accuracy, label="test accuracy")
plt.ylabel("Accuracy")
plt.xlabel("n_neighbors")
plt.legend()
plt.title("Figure 2-8: Comparing training and test accuracy")
plt.show()

### 5.2 KNN Regression

In [ ]:
mglearn.plots.plot_knn_regression(n_neighbors=1)
plt.title("Figure 2-9: Predictions made by the one-nearest-neighbor regression model")
plt.show()

In [ ]:
mglearn.plots.plot_knn_regression(n_neighbors=3)
plt.title("Figure 2-10: Predictions made by the three-nearest-neighbors regression model")
plt.show()

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

X, y = mglearn.datasets.make_wave(n_samples=40)

# split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# instantiate the model and set the number of neighbors to consider to 3
reg = KNeighborsRegressor(n_neighbors=3)
# fit the model using the training data and training targets
reg.fit(X_train, y_train)

print("Test set predictions:\n{}".format(reg.predict(X_test)))
print("Test set R^2: {:.2f}".format(reg.score(X_test, y_test)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# create 1,000 data points, evenly spaced between -3 and 3
line = np.linspace(-3, 3, 1000).reshape(-1, 1)
for n_neighbors, ax in zip([1, 3, 9], axes):
    # make predictions using 1, 3, or 9 neighbors
    reg = KNeighborsRegressor(n_neighbors=n_neighbors)
    reg.fit(X_train, y_train)
    ax.plot(line, reg.predict(line))
    ax.plot(X_train, y_train, '^', c=mglearn.cm2(0), markersize=8)
    ax.plot(X_test, y_test, 'v', c=mglearn.cm2(1), markersize=8)
    ax.set_title(
        "{} neighbor(s)\n train score: {:.2f} test score: {:.2f}".format(
            n_neighbors, reg.score(X_train, y_train),
            reg.score(X_test, y_test)))
    ax.set_xlabel("Feature")
    ax.set_ylabel("Target")
axes[0].legend(["Model predictions", "Training data/target", "Test data/target"], loc="best")
plt.suptitle("Figure 2-11: Comparing predictions of KNN regression with different n_neighbors")
plt.tight_layout()
plt.show()

## 6. Linear Models

Model linier membuat prediksi menggunakan fungsi linier dari fitur input.
Formula umum: `ŷ = w[0] * x[0] + w[1] * x[1] + ... + w[p] * x[p] + b`


### 6.1 Linear Regression (Ordinary Least Squares)

In [ ]:
from sklearn.linear_model import LinearRegression
X, y = mglearn.datasets.make_wave(n_samples=60)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

lr = LinearRegression().fit(X_train, y_train)

print("lr.coef_: {}".format(lr.coef_))
print("lr.intercept_: {}".format(lr.intercept_))
print("Training set score: {:.2f}".format(lr.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lr.score(X_test, y_test)))

In [ ]:
# Using extended boston dataset
X, y = mglearn.datasets.load_extended_boston()
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
lr = LinearRegression().fit(X_train, y_train)

print("Training set score: {:.2f}".format(lr.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lr.score(X_test, y_test)))

### 6.2 Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge().fit(X_train, y_train)
print("Training set score: {:.2f}".format(ridge.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge.score(X_test, y_test)))

In [ ]:
# Efek alpha pada Ridge
ridge10 = Ridge(alpha=10).fit(X_train, y_train)
print("Ridge alpha=10")
print("Training set score: {:.2f}".format(ridge10.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge10.score(X_test, y_test)))

ridge01 = Ridge(alpha=0.1).fit(X_train, y_train)
print("\nRidge alpha=0.1")
print("Training set score: {:.2f}".format(ridge01.score(X_train, y_train)))
print("Test set score: {:.2f}".format(ridge01.score(X_test, y_test)))

In [ ]:
# Perbandingan koefisien Ridge vs Linear Regression
plt.plot(ridge.coef_, 's', label="Ridge alpha=1")
plt.plot(ridge10.coef_, '^', label="Ridge alpha=10")
plt.plot(ridge01.coef_, 'v', label="Ridge alpha=0.1")
plt.plot(lr.coef_, 'o', label="LinearRegression")
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
plt.hlines(0, 0, len(lr.coef_))
plt.ylim(-25, 25)
plt.legend()
plt.title("Figure 2-12: Comparing coefficients of Ridge vs LinearRegression")
plt.show()

In [ ]:
# Learning curves - Ridge vs Linear Regression
mglearn.plots.plot_ridge_n_samples()
plt.title("Figure 2-13: Ridge vs LinearRegression as function of training set size")
plt.show()

### 6.3 Lasso

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso().fit(X_train, y_train)
print("Training set score: {:.2f}".format(lasso.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lasso.score(X_test, y_test)))
print("Number of features used: {}".format(np.sum(lasso.coef_ != 0)))

In [ ]:
# Lasso dengan alpha lebih kecil - perlu max_iter lebih besar
lasso001 = Lasso(alpha=0.01, max_iter=100000).fit(X_train, y_train)
print("Lasso alpha=0.01")
print("Training set score: {:.2f}".format(lasso001.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lasso001.score(X_test, y_test)))
print("Number of features used: {}".format(np.sum(lasso001.coef_ != 0)))

lasso00001 = Lasso(alpha=0.0001, max_iter=100000).fit(X_train, y_train)
print("\nLasso alpha=0.0001")
print("Training set score: {:.2f}".format(lasso00001.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lasso00001.score(X_test, y_test)))
print("Number of features used: {}".format(np.sum(lasso00001.coef_ != 0)))

In [ ]:
plt.plot(lasso.coef_, 's', label="Lasso alpha=1")
plt.plot(lasso001.coef_, '^', label="Lasso alpha=0.01")
plt.plot(lasso00001.coef_, 'v', label="Lasso alpha=0.0001")
plt.plot(ridge01.coef_, 'o', label="Ridge alpha=0.1")
plt.legend(ncol=2, loc=(0, 1.05))
plt.ylim(-25, 25)
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
plt.title("Figure 2-14: Comparing Lasso coefficients for different alpha")
plt.show()

### 6.4 Linear Models for Classification

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

X, y = mglearn.datasets.make_forge()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for model, ax in zip([LinearSVC(max_iter=5000), LogisticRegression()], axes):
    clf = model.fit(X, y)
    mglearn.plots.plot_2d_separator(clf, X, fill=False, eps=0.5, ax=ax, alpha=.7)
    mglearn.discrete_scatter(X[:, 0], X[:, 1], y, ax=ax)
    ax.set_title("{}".format(clf.__class__.__name__))
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")
axes[0].legend()
plt.suptitle("Figure 2-15: Decision boundaries of LinearSVC and LogisticRegression")
plt.tight_layout()
plt.show()

In [ ]:
# Efek parameter C pada LogisticRegression
mglearn.plots.plot_linear_svc_regularization()
plt.title("Figure 2-16: Decision boundaries for different values of C in LinearSVC")
plt.show()

In [ ]:
# LogisticRegression pada Cancer dataset
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

logreg = LogisticRegression(max_iter=10000).fit(X_train, y_train)
print("Training set score: {:.3f}".format(logreg.score(X_train, y_train)))
print("Test set score: {:.3f}".format(logreg.score(X_test, y_test)))

In [ ]:
logreg100 = LogisticRegression(C=100, max_iter=10000).fit(X_train, y_train)
print("C=100")
print("Training set score: {:.3f}".format(logreg100.score(X_train, y_train)))
print("Test set score: {:.3f}".format(logreg100.score(X_test, y_test)))

logreg001 = LogisticRegression(C=0.01, max_iter=10000).fit(X_train, y_train)
print("\nC=0.01")
print("Training set score: {:.3f}".format(logreg001.score(X_train, y_train)))
print("Test set score: {:.3f}".format(logreg001.score(X_test, y_test)))

In [ ]:
# Visualisasi koefisien LogisticRegression
plt.plot(logreg.coef_.T, 'o', label="C=1")
plt.plot(logreg100.coef_.T, '^', label="C=100")
plt.plot(logreg001.coef_.T, 'v', label="C=0.001")
plt.xticks(range(cancer.data.shape[1]), cancer.feature_names, rotation=90)
plt.hlines(0, 0, cancer.data.shape[1])
plt.ylim(-5, 5)
plt.xlabel("Feature")
plt.ylabel("Coefficient magnitude")
plt.legend()
plt.title("Figure 2-17: Coefficients of LogisticRegression for different C values")
plt.tight_layout()
plt.show()

#### Linear Models for Multiclass Classification

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(random_state=42)

mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.legend(["Class 0", "Class 1", "Class 2"])
plt.title("Multiclass dataset")
plt.show()

In [ ]:
linear_svm = LinearSVC(max_iter=10000).fit(X, y)
print("Coefficient shape: ", linear_svm.coef_.shape)
print("Intercept shape: ", linear_svm.intercept_.shape)

In [ ]:
mglearn.plots.plot_2d_classification(linear_svm, X, fill=True, alpha=.7)
mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
line = np.linspace(-15, 15)
for coef, intercept, color in zip(linear_svm.coef_, linear_svm.intercept_,
                                   mglearn.cm3.colors):
    plt.plot(line, -(line * coef[0] + intercept) / coef[1], c=color)
plt.legend(['Class 0', 'Class 1', 'Class 2', 'Line class 0',
            'Line class 1', 'Line class 2'], loc=(1.01, 0.3))
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Figure 2-18: Decision boundaries for multiclass classification")
plt.show()

## 7. Naive Bayes Classifiers

Naive Bayes adalah kelompok classifier yang mirip dengan linear models tapi training lebih cepat.
Asumsi: setiap fitur bersifat independen satu sama lain.

Tiga varian utama:
- `GaussianNB`: untuk fitur kontinu
- `BernoulliNB`: untuk fitur biner
- `MultinomialNB`: untuk fitur berupa hitungan (count)


In [ ]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
import numpy as np

# Contoh sederhana dengan GaussianNB
X_train = np.array([[0, 1, 0, 1],
                    [1, 0, 1, 1],
                    [0, 0, 0, 1],
                    [1, 1, 1, 0]])
y_train = np.array([0, 1, 0, 1])

# BernoulliNB - menghitung berapa kali tiap fitur bernilai non-zero
clf = BernoulliNB()
clf.fit(X_train, y_train)

# Probabilitas setiap fitur untuk setiap kelas
print("Feature probability per class (log-scale):")
print(clf.feature_log_prob_)

In [ ]:
# GaussianNB pada dataset cancer
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
print("Training set score: {:.3f}".format(gnb.score(X_train, y_train)))
print("Test set score: {:.3f}".format(gnb.score(X_test, y_test)))

## 8. Decision Trees

Decision Tree adalah struktur hierarki yang membuat keputusan berdasarkan aturan if-then.
Model ini bisa dengan mudah menjadi overfit jika tidak dibatasi.


In [ ]:
mglearn.plots.plot_animal_tree()
plt.title("Figure 2-22: Example of a decision tree for differentiating animals")
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))

In [ ]:
# Membatasi kedalaman tree (pre-pruning)
tree = DecisionTreeClassifier(max_depth=4, random_state=0)
tree.fit(X_train, y_train)

print("Accuracy on training set: {:.3f}".format(tree.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(tree.score(X_test, y_test)))

In [ ]:
# Visualisasi Decision Tree
from sklearn.tree import export_graphviz
import graphviz

try:
    export_graphviz(tree, out_file="tree.dot", class_names=["malignant", "benign"],
                    feature_names=cancer.feature_names, impurity=False, filled=True)
    with open("tree.dot") as f:
        dot_graph = f.read()
    display(graphviz.Source(dot_graph))
    print("Visualisasi Decision Tree berhasil ditampilkan")
except Exception as e:
    print(f"Graphviz tidak tersedia: {e}")
    print("Alternatif: menggunakan sklearn.tree.plot_tree")
    from sklearn.tree import plot_tree
    plt.figure(figsize=(20, 10))
    plot_tree(tree, class_names=["malignant", "benign"],
              feature_names=cancer.feature_names, impurity=False,
              filled=True, max_depth=3, fontsize=8)
    plt.title("Decision Tree Visualization (max_depth=4, shown up to depth 3)")
    plt.show()

In [ ]:
# Feature Importance
print("Feature importances:\n{}".format(tree.feature_importances_))

In [ ]:
# Visualisasi feature importance
def plot_feature_importances_cancer(model, dataset):
    n_features = dataset.data.shape[1]
    plt.barh(range(n_features), model.feature_importances_, align='center')
    plt.yticks(np.arange(n_features), dataset.feature_names)
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.ylim(-1, n_features)

plot_feature_importances_cancer(tree, cancer)
plt.title("Figure 2-24: Feature importances of decision tree on cancer dataset")
plt.tight_layout()
plt.show()

In [ ]:
# Decision Tree untuk Regresi
from sklearn.tree import DecisionTreeRegressor

# Data penggunaan RAM - contoh dari buku
import os
ram_prices = pd.read_csv("https://raw.githubusercontent.com/amueller/introduction_to_ml_with_python/master/data/ram_price.csv")

plt.semilogy(ram_prices.date, ram_prices.price)
plt.xlabel("Year")
plt.ylabel("Price in $/Mbyte")
plt.title("Figure 2-25: Historical prices of RAM")
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# use historical data to forecast prices after the year 2000
data_train = ram_prices[ram_prices.date < 2000]
data_test = ram_prices[ram_prices.date >= 2000]

# predict prices based on date
X_train = data_train.date.to_numpy().reshape(-1, 1)
y_train = np.log(data_train.price)  # use log-transform

X_test = data_test.date.to_numpy().reshape(-1, 1)
y_test = np.log(data_test.price)

tree = DecisionTreeRegressor().fit(X_train, y_train)
linear_reg = LinearRegression().fit(X_train, y_train)

# predict on full date range
X_all = ram_prices.date.to_numpy().reshape(-1, 1)

pred_tree = tree.predict(X_all)
pred_lr = linear_reg.predict(X_all)

# undo log-transform
price_tree = np.exp(pred_tree)
price_lr = np.exp(pred_lr)

plt.semilogy(data_train.date, data_train.price, label="Training data")
plt.semilogy(data_test.date, data_test.price, label="Test data")
plt.semilogy(ram_prices.date, price_tree, label="Tree predictions")
plt.semilogy(ram_prices.date, price_lr, label="Linear predictions")
plt.legend()
plt.title("Figure 2-26: Comparison of Decision Tree vs Linear Regression predictions on RAM prices")
plt.xlabel("Year")
plt.ylabel("Price in $/Mbyte")
plt.show()

## 9. Ensembles of Decision Trees

Ensemble methods menggabungkan banyak model ML untuk membuat model yang lebih kuat.
Dua metode ensemble paling populer:
1. **Random Forests** - bagging dengan random feature selection
2. **Gradient Boosting** - boosting sequential


### 9.1 Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=100, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

forest = RandomForestClassifier(n_estimators=5, random_state=2)
forest.fit(X_train, y_train)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for i, (ax, tree) in enumerate(zip(axes.ravel(), forest.estimators_)):
    ax.set_title("Tree {}".format(i))
    mglearn.plots.plot_tree_partition(X_train, y_train, tree, ax=ax)

mglearn.plots.plot_2d_separator(forest, X_train, fill=True, ax=axes[-1, -1], alpha=.4)
axes[-1, -1].set_title("Random Forest")
mglearn.discrete_scatter(X_train[:, 0], X_train[:, 1], y_train)
plt.suptitle("Figure 2-31: Trees built by random forest (5 trees + combined)")
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest pada Cancer dataset
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_train, y_train)

print("Accuracy on training set: {:.3f}".format(forest.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(forest.score(X_test, y_test)))

In [ ]:
# Feature Importance dari Random Forest
plot_feature_importances_cancer(forest, cancer)
plt.title("Figure 2-33: Feature importances of random forest on cancer dataset")
plt.tight_layout()
plt.show()

### 9.2 Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

gbrt = GradientBoostingClassifier(random_state=0)
gbrt.fit(X_train, y_train)

print("Accuracy on training set: {:.3f}".format(gbrt.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(gbrt.score(X_test, y_test)))

In [ ]:
# Mencegah overfitting dengan max_depth
gbrt = GradientBoostingClassifier(random_state=0, max_depth=1)
gbrt.fit(X_train, y_train)

print("max_depth=1")
print("Accuracy on training set: {:.3f}".format(gbrt.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(gbrt.score(X_test, y_test)))

gbrt = GradientBoostingClassifier(random_state=0, learning_rate=0.01)
gbrt.fit(X_train, y_train)
print("\nlearning_rate=0.01")
print("Accuracy on training set: {:.3f}".format(gbrt.score(X_train, y_train)))
print("Accuracy on test set: {:.3f}".format(gbrt.score(X_test, y_test)))

In [ ]:
# Feature importance GradientBoosting
gbrt = GradientBoostingClassifier(random_state=0, max_depth=1)
gbrt.fit(X_train, y_train)

plot_feature_importances_cancer(gbrt, cancer)
plt.title("Figure 2-35: Feature importances from gradient boosting classifier")
plt.tight_layout()
plt.show()

## 10. Kernelized Support Vector Machines (SVM)

SVM dengan kernel trick memungkinkan pembuatan decision boundary yang lebih kompleks.
Kernel yang umum digunakan:
- **Linear kernel** (sama dengan LinearSVC)
- **RBF (Radial Basis Function) kernel** – paling umum
- **Polynomial kernel**


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_blobs

# Dataset untuk visualisasi SVM
X, y = make_blobs(centers=4, random_state=8)
y = y % 2  # Buat hanya 2 kelas

mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Binary classification dataset")
plt.show()

In [ ]:
# Linear SVM tidak cukup untuk data ini
from sklearn.svm import LinearSVC

linear_svm = LinearSVC(max_iter=5000).fit(X, y)
mglearn.plots.plot_2d_separator(linear_svm, X)
mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("Linear SVM decision boundary")
plt.show()

In [ ]:
# Menambah fitur baru (kuadrat) untuk memungkinkan separasi linear
X_new = np.hstack([X, X[:, 1:] ** 2])

from mpl_toolkits.mplot3d import Axes3D, axes3d

figure = plt.figure(figsize=(12, 4))
# visualize in 3D
ax = Axes3D(figure, elev=-152, azim=-26)
# plot first all the points with y == 0, then all with y == 1
mask = y == 0
ax.scatter(X_new[mask, 0], X_new[mask, 1], X_new[mask, 2], c='b',
           cmap=mglearn.cm2, s=60, edgecolor='k')
ax.scatter(X_new[~mask, 0], X_new[~mask, 1], X_new[~mask, 2], c='r', marker='^',
           cmap=mglearn.cm2, s=60, edgecolor='k')
ax.set_xlabel("feature0")
ax.set_ylabel("feature1")
ax.set_zlabel("feature1 ** 2")
plt.title("Figure 2-37: 3D visualization of expanded feature set")
plt.show()

In [ ]:
# SVM dengan RBF kernel
X, y = make_blobs(centers=4, random_state=8)
y = y % 2

svm = SVC(kernel='rbf', C=10, gamma=0.1).fit(X, y)
mglearn.plots.plot_2d_separator(svm, X, eps=.5)
mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
# plot support vectors
sv = svm.support_vectors_
# class labels of support vectors are given by the sign of the dual coefficients
sv_labels = svm.dual_coef_.ravel() > 0
mglearn.discrete_scatter(sv[:, 0], sv[:, 1], sv_labels, s=15, markeredgewidth=3)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("SVM with RBF kernel")
plt.show()

In [ ]:
# Efek parameter C dan gamma pada SVM
fig, axes = plt.subplots(3, 3, figsize=(15, 10))

for ax, C in zip(axes, [-1, 0, 3]):
    for a, gamma in zip(ax, range(-1, 2)):
        mglearn.plots.plot_svm(log_C=C, log_gamma=gamma, ax=a)

axes[0, 0].legend(["class 0", "class 1", "sv class 0", "sv class 1"],
                  ncol=4, loc=(.9, 1.2))
plt.suptitle("Figure 2-42: Decision boundaries and support vectors for different C and gamma")
plt.tight_layout()
plt.show()

In [ ]:
# SVM pada Breast Cancer dataset
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

svc = SVC()
svc.fit(X_train, y_train)

print("Accuracy on training set: {:.2f}".format(svc.score(X_train, y_train)))
print("Accuracy on test set: {:.2f}".format(svc.score(X_test, y_test)))

In [ ]:
# Melihat range fitur - penyebab performa SVM buruk
plt.plot(X_train.min(axis=0), 'o', label='min')
plt.plot(X_train.max(axis=0), '^', label='max')
plt.legend(loc=4)
plt.xlabel("Feature index")
plt.ylabel("Feature magnitude")
plt.yscale("log")
plt.title("Figure 2-43: Feature ranges in the cancer dataset (log scale)")
plt.show()

In [ ]:
# Preprocessing - scaling data untuk SVM
# Compute the minimum value per feature on the training set
min_on_training = X_train.min(axis=0)
# Compute the range of each feature (max - min) on the training set
range_on_training = (X_train - min_on_training).max(axis=0)

# subtract the min, and divide by range
# afterward, min=0 and max=1 for each feature
X_train_scaled = (X_train - min_on_training) / range_on_training
print("Minimum for each feature\n{}".format(X_train_scaled.min(axis=0)))
print("Maximum for each feature\n {}".format(X_train_scaled.max(axis=0)))

In [ ]:
# use THE SAME transformation on the test set,
# using min and range of the training set (see Chapter 3 for details)
X_test_scaled = (X_test - min_on_training) / range_on_training

svc = SVC()
svc.fit(X_train_scaled, y_train)

print("Accuracy on training set: {:.3f}".format(svc.score(X_train_scaled, y_train)))
print("Accuracy on test set: {:.3f}".format(svc.score(X_test_scaled, y_test)))

In [ ]:
# SVM dengan C lebih besar setelah scaling
svc = SVC(C=1000)
svc.fit(X_train_scaled, y_train)

print("Accuracy on training set: {:.3f}".format(svc.score(X_train_scaled, y_train)))
print("Accuracy on test set: {:.3f}".format(svc.score(X_test_scaled, y_test)))

## 11. Neural Networks (Deep Learning)

Neural networks (Multi-layer Perceptron / MLP) adalah model komputasi yang terinspirasi dari otak manusia.
Terdiri dari lapisan-lapisan neuron (node) yang terhubung.

Fungsi aktivasi yang umum digunakan: **relu** (default) dan **tanh**


In [ ]:
# Visualisasi arsitektur neural network
display(mglearn.plots.plot_logistic_regression_graph())

In [ ]:
display(mglearn.plots.plot_single_hidden_layer_graph())

In [ ]:
# Efek fungsi aktivasi
line = np.linspace(-3, 3, 100)
plt.plot(line, np.tanh(line), label="tanh")
plt.plot(line, np.maximum(line, 0), label="relu")
plt.legend(loc="best")
plt.xlabel("x")
plt.ylabel("relu(x), tanh(x)")
plt.title("Figure 2-45: relu and tanh activation functions")
plt.show()

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=100, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

mlp = MLPClassifier(solver='lbfgs', random_state=0).fit(X_train, y_train)
mglearn.plots.plot_2d_separator(mlp, X_train, fill=True, alpha=.3)
mglearn.discrete_scatter(X_train[:, 0], X_train[:, 1], y_train)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.title("MLP decision boundary on moons dataset")
plt.show()

In [ ]:
# Efek n_hidden_units dan aktivasi
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for axx, n_hidden_nodes in zip(axes, [10, 100]):
    for ax, activation in zip(axx, ['relu', 'tanh']):
        mlp = MLPClassifier(solver='lbfgs', random_state=0,
                           hidden_layer_sizes=[n_hidden_nodes],
                           activation=activation)
        mlp.fit(X_train, y_train)
        mglearn.plots.plot_2d_separator(mlp, X_train, fill=True, alpha=.3, ax=ax)
        mglearn.discrete_scatter(X_train[:, 0], X_train[:, 1], y_train, ax=ax)
        ax.set_title("n_hidden=[{}],\nactivation={}".format(n_hidden_nodes, activation))
plt.suptitle("Figure 2-46: Decision boundaries with different hidden nodes and activations")
plt.tight_layout()
plt.show()

In [ ]:
# MLP dengan berbagai kedalaman (multiple hidden layers)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for axx, n_hidden_nodes in zip(axes, [10, 100]):
    for ax, n_layers in zip(axx, range(1, 5)):
        mlp = MLPClassifier(solver='lbfgs', random_state=0,
                           hidden_layer_sizes=[n_hidden_nodes] * n_layers)
        mlp.fit(X_train, y_train)
        mglearn.plots.plot_2d_separator(mlp, X_train, fill=True, alpha=.3, ax=ax)
        mglearn.discrete_scatter(X_train[:, 0], X_train[:, 1], y_train, ax=ax)
        ax.set_title("n_hidden=[{}]*{}".format(n_hidden_nodes, n_layers))
plt.suptitle("Figure 2-47: Decision boundaries with different layer depths")
plt.tight_layout()
plt.show()

In [ ]:
# MLP pada Cancer dataset
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

mlp = MLPClassifier(random_state=42)
mlp.fit(X_train, y_train)

print("Accuracy on training set: {:.2f}".format(mlp.score(X_train, y_train)))
print("Accuracy on test set: {:.2f}".format(mlp.score(X_test, y_test)))

In [ ]:
# Scaling data untuk MLP
# Compute the mean value per feature on the training set
mean_on_train = X_train.mean(axis=0)
# Compute the standard deviation of each feature on the training set
std_on_train = X_train.std(axis=0)

# subtract the mean, scale by inverse standard deviation
# afterward, mean=0 and std=1
X_train_scaled = (X_train - mean_on_train) / std_on_train
# use THE SAME transformation on the test set
X_test_scaled = (X_test - mean_on_train) / std_on_train

mlp = MLPClassifier(random_state=0)
mlp.fit(X_train_scaled, y_train)

print("Accuracy on training set: {:.3f}".format(mlp.score(X_train_scaled, y_train)))
print("Accuracy on test set: {:.3f}".format(mlp.score(X_test_scaled, y_test)))

In [ ]:
# MLP dengan lebih banyak iterasi
mlp = MLPClassifier(max_iter=1000, random_state=0)
mlp.fit(X_train_scaled, y_train)

print("Accuracy on training set: {:.3f}".format(mlp.score(X_train_scaled, y_train)))
print("Accuracy on test set: {:.3f}".format(mlp.score(X_test_scaled, y_test)))

In [ ]:
# Visualisasi bobot MLP - heatmap
plt.figure(figsize=(20, 5))
plt.imshow(mlp.coefs_[0], interpolation='none', cmap='viridis')
plt.yticks(range(30), cancer.feature_names)
plt.xlabel("Columns in weight matrix")
plt.ylabel("Input feature")
plt.colorbar()
plt.title("Figure 2-49: Heatmap of first layer weights of MLPClassifier on cancer dataset")
plt.show()

## 12. Uncertainty Estimates from Classifiers

Scikit-learn menyediakan dua cara untuk memperoleh estimasi ketidakpastian prediksi:
1. **decision_function** - skor kepercayaan per kelas (bisa negatif/positif)
2. **predict_proba** - probabilitas per kelas (antara 0-1, jumlah = 1)


### 12.1 The Decision Function

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_circles

X, y = make_circles(noise=0.25, factor=0.5, random_state=1)

# rename classes "blue" and "red" for illustration purposes
y_named = np.array(["blue", "red"])[y]

# we can call train_test_split with arbitrarily many arrays
# all will be split consistently
X_train, X_test, y_train_named, y_test_named, y_train, y_test =     train_test_split(X, y_named, y, random_state=0)

# build the gradient boosting model
gbrt = GradientBoostingClassifier(random_state=0)
gbrt.fit(X_train, y_train_named)

print("X_test.shape: {}".format(X_test.shape))
print("Decision function shape: {}".format(
    gbrt.decision_function(X_test).shape))

In [ ]:
# show the first few entries of decision_function
print("Decision function:\n{}".format(gbrt.decision_function(X_test)[:6]))

In [ ]:
print("Thresholded decision function:\n{}".format(
    gbrt.decision_function(X_test) > 0))
print("Predictions:\n{}".format(gbrt.predict(X_test)))

In [ ]:
# make the boolean True/False into 0 and 1
greater_zero = (gbrt.decision_function(X_test) > 0).astype(int)
# use 0 and 1 as indices into classes_
pred = gbrt.classes_[greater_zero]
# pred is the same as the output of gbrt.predict
print("pred is same as predict: {}".format(
    np.all(pred == gbrt.predict(X_test))))

In [ ]:
# Visualisasi decision function
decision_function = gbrt.decision_function(X_test)

print("Decision function minimum: {:.2f} maximum: {:.2f}".format(
    decision_function.min(), decision_function.max()))

plt.figure(figsize=(10, 3))
colors = ["blue" if c == 1 else "red" for c in y_test]
plt.scatter(range(len(X_test)), decision_function, c=colors, s=60)
plt.xticks(())
plt.ylabel("Decision function")
plt.title("Figure 2-50: Decision function for binary classification")
plt.show()

### 12.2 Predicting Probabilities

In [ ]:
print("Shape of probabilities: {}".format(gbrt.predict_proba(X_test).shape))

In [ ]:
# show the first few entries of predict_proba
print("Predicted probabilities:")
print(gbrt.predict_proba(X_test[:6]))

In [ ]:
plt.figure(figsize=(10, 3))
plt.scatter(range(len(X_test)), gbrt.predict_proba(X_test)[:, 1], c=colors, s=60)
plt.xticks(())
plt.ylabel('Predicted probability of class "red"')
plt.title('Figure 2-51: Predicted probabilities for binary classification')
plt.show()

### 12.3 Uncertainty in Multiclass Classification

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, random_state=42)

gbrt = GradientBoostingClassifier(learning_rate=0.01, random_state=0)
gbrt.fit(X_train, y_train)

print("Decision function shape: {}".format(gbrt.decision_function(X_test).shape))

In [ ]:
# plot the first few entries of the decision function
print("Decision function:\n{}".format(gbrt.decision_function(X_test)[:6, :]))

In [ ]:
print("Argmax of decision function:\n{}".format(
    np.argmax(gbrt.decision_function(X_test), axis=1)))
print("Predictions:\n{}".format(gbrt.predict(X_test)))

In [ ]:
# predict_proba untuk multiclass
print("Predicted probabilities:\n{}".format(gbrt.predict_proba(X_test)[:6]))

In [ ]:
print("Sums: {}".format(gbrt.predict_proba(X_test)[:6].sum(axis=1)))

In [ ]:
print("Argmax of predicted probabilities:\n{}".format(
    np.argmax(gbrt.predict_proba(X_test), axis=1)))
print("Predictions:\n{}".format(gbrt.predict(X_test)))

In [ ]:
logreg = LogisticRegression(max_iter=10000)

# represent each target by its class name in the iris dataset
named_target = iris.target_names[y_train]
logreg.fit(X_train, named_target)
print("unique classes in training data: {}".format(logreg.classes_))
print("predictions: {}".format(logreg.predict(X_test)[:10]))
argmax_dec_func = np.argmax(logreg.decision_function(X_test), axis=1)
print("argmax of decision function: {}".format(argmax_dec_func[:10]))
print("argmax combined with classes_: {}".format(
    logreg.classes_[argmax_dec_func][:10]))

## 13. Summary and Outlook

### Ringkasan Algoritma Supervised Learning

| Algoritma | Kelebihan | Kekurangan |
|-----------|-----------|------------|
| **KNN** | Sederhana, intuitif | Lambat pada data besar, sensitif terhadap skala |
| **Linear Models** | Cepat, interpretable, baik untuk high-dim data | Asumsi linieritas |
| **Naive Bayes** | Sangat cepat, bekerja baik pada sparse data | Asumsi independence |
| **Decision Tree** | Interpretable, tidak perlu scaling | Mudah overfit |
| **Random Forest** | Performa baik, robust, parallelizable | Sulit diinterpretasi |
| **Gradient Boosting** | Sering performa terbaik | Lambat di training, banyak hyperparameter |
| **SVM** | Bagus untuk high-dim, memory efficient | Perlu feature scaling, sulit interpretasi |
| **MLP/Neural Net** | Dapat belajar fitur kompleks | Banyak hyperparameter, perlu scaling, butuh data banyak |

### Panduan Pemilihan Model

- **Data kecil, features sedikit**: Linear models, Naive Bayes
- **Data sedang**: Random Forest, Gradient Boosting
- **Data besar, data tidak terstruktur**: Neural Networks
- **Perlu interpretasi**: Linear Models, Decision Trees
